# Generative predictive control

A conditional flow policy learns whole control sequences from a sampling planner. This guide trains a policy to drive a scalar integrator to the origin, then compares direct GPC inference with GPC+, which refines the decision using the same factor graph that supplied the training targets. Frank Dellaert and Akash Ratheesh.

GTSAM Copyright 2010–2026, Georgia Tech Research Corporation, Atlanta, Georgia 30332-0415. All rights reserved. See LICENSE for details.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam_unstable/planning/doc/GenerativePredictiveControl.ipynb)

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop plotly
except ImportError:
    pass

In [2]:
import numpy as np
import gtsam
import gtsam_unstable as unstable
import plotly.graph_objects as go
from IPython.display import display

## Specify the teacher's objective

For the model $x_{k+1}=x_k+\Delta t\,u_k$, a prior on $X_H$ sets the goal. Priors on the control keys penalize effort. Each batch has one row per action dimension and one block of $H$ consecutive columns per candidate.

In [3]:
H, dt = 4, 0.5
states = [gtsam.symbol("x", k) for k in range(H + 1)]
controls = [gtsam.symbol("u", k) for k in range(H)]
graph = gtsam.NonlinearFactorGraph()
goal = gtsam.noiseModel.Isotropic.Sigma(1, 0.15)
effort = gtsam.noiseModel.Isotropic.Sigma(1, 1.5)
graph.add(gtsam.PriorFactorVector(states[-1], np.zeros(1), goal))
for key in controls:
    graph.add(gtsam.PriorFactorVector(key, np.zeros(1), effort))

def advance(x, u):
    return x + dt * u

def trajectory(x, sequence):
    values = gtsam.Values()
    values.insert(states[0], x)
    for k, u in enumerate(sequence.T):
        x = advance(x, u)
        values.insert(controls[k], u)
        values.insert(states[k + 1], x)
    return values

def score_batch(x, batch):
    return np.array([graph.error(trajectory(x, batch[:, k:k + H]))
                     for k in range(0, batch.shape[1], H)])

## Alternate planning and flow matching

Each cycle starts several episodes. At each step, the teacher scores a mixture of Gaussian perturbations and samples from the current flow policy, selects the lowest-cost trajectory, and applies its first action. The controller collects the observation, the incoming plan, and the improved plan.

The supervised target for a training source $Z\sim\mathcal N(0,I)$ and improved sequence $U$ is the velocity $U-Z$ at the interpolated input $(1-t)Z+tU$. The network also receives the observation and flow time $t$. The cosine weighting parameter emphasizes training draws aligned with the planner's improvement. [Kurtz and Burdick, Algorithm 1 and Appendix I](https://arxiv.org/abs/2502.13406)

In [4]:
model = unstable.GenerativePredictiveControl(1, 1, H, 24, 17)
params = unstable.GPCTrainingParams()
params.episodeLength = 6
params.gaussianSamples = 16
params.policySamples = 4
params.gradientSteps = 120
params.batchSize = 16
params.learningRate = 0.015
params.cosineWeight = 1.0
params.flowSteps = 6
teacher = gtsam.PredictiveSampling()

# Each column is the initial state of one controlled episode.
starts = np.array([[-1.5, -1.0, -0.5, 0.5, 1.0, 1.5]])
for cycle in range(3):
    loss = model.trainCycle(starts, params, lambda x: x, advance,
                            score_batch, teacher.weights)
    assert np.isfinite(loss)
    print(f"Cycle {cycle + 1}, final minibatch loss {loss:.3f}")

Cycle 1, final minibatch loss 3.249
Cycle 2, final minibatch loss 3.225
Cycle 3, final minibatch loss 2.898


## Deploy GPC and GPC+

Direct GPC integrates the learned vector field to generate controls. A warm-start level $\alpha$ sets its initial condition to $(1-\alpha)Z+\alpha U_{\mathrm{previous}}$. GPC+ adds flow samples to a Gaussian proposal and evaluates them with the planner. This example aligns successive horizons by shifting one step and holding the last action at the tail. Both controllers replan after applying the first control.

In [5]:
planner = gtsam.SamplingPlannerVectorVector(states, controls, dt)

def run_closed_loop(refine):
    x = np.array([1.25])
    previous = np.zeros((1, H))
    positions = [x[0]]
    for k in range(12):
        if refine:
            proposal = unstable.GPCProposal(model, x, np.eye(1), 16, 4, 0.7, 6, 50 + k)
            plan = planner.optimize(graph, trajectory(x, previous), lambda x, u: u,
                                    proposal, teacher, 2)
            sequence = np.column_stack([plan.atVector(key) for key in controls])
        else:
            sequence = model.generate(x, previous, 0.7, 6)
        x = advance(x, sequence[:, 0])
        positions.append(x[0])
        previous = np.column_stack([sequence[:, 1:], sequence[:, -1]])
    return np.array(positions)

direct = run_closed_loop(False)
refined = run_closed_loop(True)
assert np.isfinite(direct).all() and np.isfinite(refined).all()
print(f"Final |x|: direct {abs(direct[-1]):.3f}, refined {abs(refined[-1]):.3f}")

Final |x|: direct 0.457, refined 0.177


The final distance measures regulation error on this rollout. The curves show how quickly the policies approach the goal and how they react near it. The interactive chart requires an internet connection.

In [6]:
fig = go.Figure()
for name, positions in [("GPC", direct), ("GPC+", refined)]:
    fig.add_scatter(x=dt * np.arange(len(positions)), y=positions,
                    mode="lines+markers", name=name)
fig.add_hline(y=0, line_dash="dot")
fig.update_layout(xaxis_title="Time (s)", yaxis_title="Position")
from html import escape

document = fig.to_html(include_plotlyjs="cdn", config={"displaylogo": False, "responsive": True})
display({'text/html': f'<iframe title="Generative predictive control" style="width:100%;border:0" '
             f'height="{fig.layout.height or 480}" srcdoc="{escape(document, quote=True)}"></iframe>'}, raw=True)

<iframe title="Generative predictive control" style="width:100%;border:0" height="480" srcdoc="<html>
<head><meta charset="utf-8" /></head>
<body>
 <div> <script type="text/javascript">window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="https://cdn.plot.ly/plotly-3.1.1.min.js" integrity="sha256-HUEFyfiTnZJxCxur99FjbKYTvKSzwDaD3/x5TqHpFu4=" crossorigin="anonymous"></script> <div id="ecb1023f-c80d-4a5c-b3e2-5faf4b5956cc" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script type="text/javascript"> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("ecb1023f-c80d-4a5c-b3e2-5faf4b5956cc")) { Plotly.newPlot( "ecb1023f-c80d-4a5c-b3e2-5faf4b5956cc", [{"mode":"lines+markers","name":"GPC","x":{"dtype":"f8","bdata":"AAAAAAAAAAAAAAAAAADgPwAAAAAAAPA\u002fAAAAAAAA+D8AAAAAAAAAQAAAAAAAAARAAAAAAAAACEAAAAAAAAAMQAAAAAAAABBAAAAAAAAAEkAAAAAAAAAUQAAAAAAAABZAAAAAAAAAGEA="},"y":{"dtype":"f8","bdata":"AAAAAAAA9D+5MPitK8TuP7cIL\u002fIP9O4\u002f0q6oJnNi6T\u002fKGwstJj3pP3OWDCKrSeg\u002f0iBWpw0d4j8cmuQHkOfhP+S052r7BOY\u002fW5M0V1KX4z9YBCs1qtDiP1CrIk9wPOM\u002f3mqQbOc93T8="},"type":"scatter"},{"mode":"lines+markers","name":"GPC+","x":{"dtype":"f8","bdata":"AAAAAAAAAAAAAAAAAADgPwAAAAAAAPA\u002fAAAAAAAA+D8AAAAAAAAAQAAAAAAAAARAAAAAAAAACEAAAAAAAAAMQAAAAAAAABBAAAAAAAAAEkAAAAAAAAAUQAAAAAAAABZAAAAAAAAAGEA="},"y":{"dtype":"f8","bdata":"AAAAAAAA9D8+d\u002fEN1TnxP7y+4b56s+w\u002fokC0BHQK7D\u002fa2vFlx2XkPw7wezRpw+E\u002f0\u002fDJyyqW4D8uCB85+jLePx0euaaJi9s\u002fNjokxjB10D\u002feRTHBGW\u002fRP7Y8eHyTvso\u002flI6XT+edxj8="},"type":"scatter"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"surface","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"mesh3d":[{"type":"mesh3d","colorbar":{"outlinewidth":0,"ticks":""}}],"scatter":[{"fillpattern":{"fillmode":"overlay","size":10,"solidity":0.2},"type":"scatter"}],"parcoords":[{"type":"parcoords","line":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolargl":[{"type":"scatterpolargl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"bar"

## Supply a policy to the generic planner

The unstable `PolicyAugmentedProposal` accepts any callable that draws an action sequence from the current observation and nominal controls. This callable uses the trained flow. The first Gaussian candidate preserves the incumbent during predictive sampling, so a refinement cannot increase the nominal plan's graph cost.

In [7]:
def draw_from_policy(initial, nominal):
    observation = initial.atVector(states[0])
    return model.generate(observation, nominal, 0.7, 6)

initial = trajectory(np.array([1.25]), np.zeros((1, H)))
mixture = unstable.PolicyAugmentedProposal(np.eye(1), 16, 4, draw_from_policy, 23)
result = planner.optimize(graph, initial, lambda x, u: u, mixture, teacher, 2)
assert graph.error(result) <= graph.error(initial)
print(f"Initial cost {graph.error(initial):.3f}, refined cost {graph.error(result):.3f}")

Initial cost 34.722, refined cost 0.477


## Score candidates across uncertain domains

To train a risk-sensitive teacher, evaluate each sequence in several simulated domains before computing the search weights. Rows below represent two candidate sequences and columns represent four equally likely domains. The upper half-tail average favors the trajectory that costs less in the two most difficult domains.

In [8]:
domain_costs = np.array([[0.2, 0.3, 0.4, 4.0], [1.0, 1.0, 1.0, 1.0]])
risk = unstable.GenerativePredictiveControl.riskCosts
print("Mean:", risk(domain_costs, 1.0))
print("Upper half-tail:", risk(domain_costs, 0.5))
assert np.allclose(risk(domain_costs, 0.5), [2.2, 1.0])

Mean: [1.225 1.   ]
Upper half-tail: [2.2 1. ]
